In [1]:
# ============================================================
#  09 Population / Residential Intensity / Activity Proxy
#  数据源: WorldPop 100m + 06 Buildings + 08 POI Demand
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
from shapely.geometry import box, mapping
from shapely.prepared import prep as shapely_prep
import requests
import warnings
warnings.filterwarnings("ignore")

RAW = Path("data_raw")
OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
BUILDINGS_06 = Path("../06 Buildings/data_out/sz_buildings.gpkg")
DEMAND_08 = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")

WORLDPOP_TIF = RAW / "chn_ppp_2020_constrained_100m.tif"
CLIPPED_TIF = OUT / "sz_worldpop_clipped.tif"

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
bbox = shenzhen_geom.bounds

print(f"06 buildings: {BUILDINGS_06.exists()}")
print(f"08 demand grid: {DEMAND_08.exists()}")
print(f"WorldPop tif: {WORLDPOP_TIF.exists()}")

06 buildings: True
08 demand grid: False
WorldPop tif: False


/Users/shirly/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
# ============================================================
#  1. 下载 WorldPop 人口栅格 (单独 cell, 只需运行一次)
#  数据: China constrained 100m, 2020
#  文件约 200-400 MB, 下载需要几分钟
#  如果已下载, 此 cell 会自动跳过
# ============================================================

WORLDPOP_URL = "https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/CHN/chn_ppp_2020_constrained.tif"

if WORLDPOP_TIF.exists():
    size_mb = WORLDPOP_TIF.stat().st_size / 1e6
    print(f"Already downloaded: {WORLDPOP_TIF} ({size_mb:.0f} MB)")
else:
    print(f"Downloading WorldPop China 100m constrained ...")
    print(f"  URL: {WORLDPOP_URL}")
    print(f"  This may take several minutes ...")

    r = requests.get(WORLDPOP_URL, stream=True, timeout=30)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))

    downloaded = 0
    with open(WORLDPOP_TIF, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                pct = downloaded / total * 100
                print(f"\r  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB ({pct:.0f}%)", end="", flush=True)

    print(f"\n  Saved -> {WORLDPOP_TIF} ({downloaded / 1e6:.0f} MB)")

In [ ]:
# ============================================================
#  2. 裁剪栅格到深圳 + 转为分析网格
#  WorldPop 100m → 裁剪 → 重采样到 0.005° (~550m) 网格
# ============================================================
from rasterio.warp import calculate_default_transform, reproject, Resampling

GRID_SIZE = 0.005  # 与 06/08 一致

# ── 裁剪 WorldPop 到深圳 ──
if not CLIPPED_TIF.exists():
    print("Clipping WorldPop to Shenzhen ...")
    with rasterio.open(WORLDPOP_TIF) as src:
        geom = [mapping(shenzhen_geom)]
        out_image, out_transform = rio_mask(src, geom, crop=True, nodata=-99999)
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": -99999,
        })
        with rasterio.open(CLIPPED_TIF, "w", **out_meta) as dst:
            dst.write(out_image)
    print(f"  Saved -> {CLIPPED_TIF}")
else:
    print(f"Clipped raster exists: {CLIPPED_TIF}")

# ── 读取裁剪后栅格 ──
with rasterio.open(CLIPPED_TIF) as src:
    pop_data = src.read(1)
    pop_transform = src.transform
    pop_crs = src.crs
    print(f"  Shape: {pop_data.shape}, CRS: {pop_crs}")
    print(f"  Valid cells: {(pop_data > 0).sum():,}")
    print(f"  Total population: {pop_data[pop_data > 0].sum():,.0f}")

# ── 转为分析网格 (0.005°) ──
minx, miny, maxx, maxy = shenzhen_geom.bounds
sz_prep = shapely_prep(shenzhen_geom)

grid_cells = []
gid = 0
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cell = box(x, y, x + GRID_SIZE, y + GRID_SIZE)
        if sz_prep.intersects(cell):
            grid_cells.append({"grid_id": gid, "geometry": cell,
                               "cx": x + GRID_SIZE / 2, "cy": y + GRID_SIZE / 2})
            gid += 1
        y += GRID_SIZE
    x += GRID_SIZE

pop_grid = gpd.GeoDataFrame(grid_cells, crs=4326)

# 对每个网格, 从栅格中采样人口 (用网格中心点)
from rasterio.transform import rowcol

with rasterio.open(CLIPPED_TIF) as src:
    pop_values = []
    for _, row in pop_grid.iterrows():
        try:
            r, c = rowcol(src.transform, row["cx"], row["cy"])
            if 0 <= r < pop_data.shape[0] and 0 <= c < pop_data.shape[1]:
                val = pop_data[r, c]
                pop_values.append(max(val, 0))
            else:
                pop_values.append(0)
        except Exception:
            pop_values.append(0)

pop_grid["pop_count"] = pop_values
grid_area_km2 = (GRID_SIZE * 111.32) ** 2
pop_grid["pop_density"] = pop_grid["pop_count"] / grid_area_km2

print(f"\nGrid: {len(pop_grid):,} cells")
print(f"Cells with population > 0: {(pop_grid['pop_count'] > 0).sum():,}")
print(f"Total grid population: {pop_grid['pop_count'].sum():,.0f}")
print(f"Max density: {pop_grid['pop_density'].max():,.0f} /km²")

In [ ]:
# ============================================================
#  3. 住宅强度 (from 06 Buildings)
#  读取已有建筑数据, 按网格汇总住宅建筑体积和栋数
# ============================================================

if BUILDINGS_06.exists():
    print("Loading buildings ...")
    bldg = gpd.read_file(BUILDINGS_06)

    # 只取住宅
    res = bldg[bldg["usage_cat"] == "residential"].copy()
    print(f"  Residential buildings: {len(res):,}")

    # 计算体积
    res["volume_m3"] = res["footprint_m2"] * res["height_m"]

    # 质心 → 空间连接到网格
    res_pts = res.copy()
    res_pts["geometry"] = res_pts.geometry.centroid
    joined = gpd.sjoin(res_pts, pop_grid[["grid_id", "geometry"]], how="inner", predicate="within")

    res_stats = joined.groupby("grid_id").agg(
        residential_count=("height_m", "count"),
        residential_volume=("volume_m3", "sum"),
        residential_avg_height=("height_m", "mean"),
    ).reset_index()

    pop_grid = pop_grid.merge(res_stats, on="grid_id", how="left")
    pop_grid[["residential_count", "residential_volume", "residential_avg_height"]] = \
        pop_grid[["residential_count", "residential_volume", "residential_avg_height"]].fillna(0)

    del bldg, res, res_pts, joined
    print(f"  Grids with residential: {(pop_grid['residential_count'] > 0).sum():,}")
else:
    print("06 Buildings not found, skipping residential intensity")
    pop_grid["residential_count"] = 0
    pop_grid["residential_volume"] = 0
    pop_grid["residential_avg_height"] = 0

In [ ]:
# ============================================================
#  4. 合并 08 活动强度 + 合成 intensity_index + 保存
# ============================================================

# ── 合并 08 demand_pressure ──
if DEMAND_08.exists():
    print("Loading 08 demand grid ...")
    demand = gpd.read_file(DEMAND_08)

    # 取 demand_pressure 列, 用网格中心点匹配
    demand["cx"] = demand.geometry.centroid.x.round(5)
    demand["cy"] = demand.geometry.centroid.y.round(5)
    pop_grid["cx_r"] = pop_grid["cx"].round(5)
    pop_grid["cy_r"] = pop_grid["cy"].round(5)

    demand_map = demand.set_index(["cx", "cy"])
    dp_col = "demand_pressure" if "demand_pressure" in demand.columns else "poi_total"

    # 空间连接 (用质心最近匹配)
    demand_pts = demand[["geometry", dp_col]].copy()
    demand_pts["geometry"] = demand_pts.geometry.centroid
    pop_pts = pop_grid[["grid_id", "geometry"]].copy()
    pop_pts["geometry"] = pop_grid.geometry.centroid

    dj = gpd.sjoin_nearest(pop_pts, demand_pts, how="left", max_distance=0.01)
    pop_grid["demand_pressure"] = dj[dp_col].fillna(0).values

    del demand, demand_pts, pop_pts, dj
    print(f"  Merged demand_pressure")
else:
    print("08 demand grid not found, skipping")
    pop_grid["demand_pressure"] = 0

# ── 合成 intensity_index ──
def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0

pop_grid["pop_norm"] = norm(pop_grid["pop_count"])
pop_grid["res_norm"] = norm(pop_grid["residential_volume"])
pop_grid["demand_norm"] = norm(pop_grid["demand_pressure"])

W_POP = 0.40
W_RES = 0.30
W_DEMAND = 0.30

pop_grid["intensity_index"] = (
    W_POP * pop_grid["pop_norm"]
  + W_RES * pop_grid["res_norm"]
  + W_DEMAND * pop_grid["demand_norm"]
)

# ── 保存 ──
out_cols = ["grid_id", "pop_count", "pop_density",
            "residential_count", "residential_volume", "residential_avg_height",
            "demand_pressure", "intensity_index", "geometry"]
result = pop_grid[[c for c in out_cols if c in pop_grid.columns]]
result.to_file(OUT / "sz_population_grid.gpkg", driver="GPKG")
print(f"\nSaved -> data_out/sz_population_grid.gpkg  ({len(result):,} cells)")

# ── 统计摘要 ──
g = result[result["pop_count"] > 0]
print(f"\n=== Summary (cells with pop > 0: {len(g):,}) ===")
print(f"Population:   total={g['pop_count'].sum():,.0f}  mean={g['pop_count'].mean():.0f}  max={g['pop_count'].max():.0f}")
print(f"Pop density:  mean={g['pop_density'].mean():,.0f} /km²  max={g['pop_density'].max():,.0f} /km²")
print(f"Res volume:   mean={g['residential_volume'].mean():,.0f} m³  max={g['residential_volume'].max():,.0f} m³")
print(f"Intensity:    mean={g['intensity_index'].mean():.3f}  max={g['intensity_index'].max():.3f}")

print("\n=== Top 10 Highest Intensity Grids ===")
top10 = result.nlargest(10, "intensity_index")[["grid_id", "pop_count", "residential_count", "demand_pressure", "intensity_index"]]
print(top10.to_string(index=False))